# part별 raw y 평균과 expected max envelope

실제 `raw.csv`의 **원래 `y_value`**를 사용합니다. 제품별로 차트를 따로 그립니다.
점 하나는 같은 STEP 안에서 특정 Order·설비·챔버 조합을 실제로 거친 웨이퍼들의 산술평균입니다.
가로축은 그 조합의 고유 웨이퍼 수 $n$, 세로축은 raw y 평균입니다.

각 part에서 다음 값만 계산합니다.

$$\hat\mu_p = \text{해당 part의 고유 웨이퍼 raw y 평균}$$
$$m_{p,n} = \text{웨이퍼 수가 정확히 }n\text{인 평균 타점 수}$$
$$\hat\sigma_{p,n} = \operatorname{std}(\bar y_{p,n,1},\ldots,\bar y_{p,n,m_{p,n}};\;ddof=1)$$
$$\text{envelope}_{p,n} = \hat\mu_p \pm \hat\sigma_{p,n}\sqrt{2\ln m_{p,n}}.$$

이미 **평균 타점들의 표준편차**를 구하므로 $\sqrt n$으로 다시 나누지 않습니다.
raw y를 중심화·표준화·순위 변환하지 않습니다. 실제 관측 조합을 전부 계산하며 무작위 추출은 하지 않습니다.
`good_bad` 값은 계산과 선별에 사용하지 않습니다. 유효한 raw y가 있는 웨이퍼는 포함합니다.

같은 part·같은 n의 평균 타점들은 독립적인 정규표본이라고 가정합니다.
이 envelope는 기대 극값의 1차 근사식으로 만든 참고선입니다. 경계 밖은 그 식을 벗어난 점이라는 뜻입니다.
$m_{p,n}=1$이면 표준편차를 계산할 수 없어 점만 표시합니다.

**사용법:** 저장소 루트에 `raw.csv`를 두고 **Run All**을 실행하세요. 다른 위치의 파일은 아래 `CSV_PATH`를 지정합니다.
필요 패키지: `numpy`, `pandas`, `matplotlib`, `ipython`. 원본 `real_data_funnel.ipynb`는 수정하지 않습니다.

In [ ]:
from pathlib import Path

# notebooks 폴더에서 실행하면 저장소 루트의 raw.csv를 읽습니다.
DATA_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
CSV_PATH = DATA_DIR / 'raw.csv'
# CSV_PATH = Path('/absolute/path/to/raw.csv')

PART_IDS = None              # None: 모든 part. 예: ['PRODUCT_A', 'PRODUCT_B']
LINE_ID = None               # 특정 line만 사용할 때 지정
STEP = None                  # None: 모든 STEP. 예: 'ex102000'
MIN_N = 20                   # 실제 조합의 최소 고유 웨이퍼 수
MAX_DEPTH = 3                # 한 조합에 묶을 Order 수: 1 ~ MAX_DEPTH
STEP_SUFFIX = r'_(\d+)$'     # step_seq 끝의 _숫자를 Order로 해석. None이면 step_ord 사용
REWORK = 'last'              # 같은 wafer·STEP·Order 재작업: 'first' 또는 'last'
LOG_X = True
OUTSIDE_ROWS = 30            # 표에 표시할 경계 밖 조합 수. 차트에는 모든 점 표시
SAVE_DIR = None              # 예: DATA_DIR / 'raw_y_funnel_results'

In [ ]:
import hashlib
import html
import math
import re
from itertools import combinations

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
from IPython.display import HTML, display

assert isinstance(MIN_N, int) and not isinstance(MIN_N, bool) and MIN_N >= 1
assert isinstance(MAX_DEPTH, int) and not isinstance(MAX_DEPTH, bool) and MAX_DEPTH >= 1
assert REWORK in ('first', 'last')
assert isinstance(OUTSIDE_ROWS, int) and OUTSIDE_ROWS >= 1

font_dirs = [Path.cwd() / 'fonts', Path.cwd() / 'notebooks' / 'fonts']
font_file = next((d / 'NanumGothic-Regular.ttf' for d in font_dirs if (d / 'NanumGothic-Regular.ttf').is_file()), None)
if font_file is not None:
    mpl.font_manager.fontManager.addfont(str(font_file))
    font_name = mpl.font_manager.FontProperties(fname=str(font_file)).get_name()
else:
    available = {f.name for f in mpl.font_manager.fontManager.ttflist}
    font_name = next((name for name in ['NanumGothic', 'Noto Sans CJK KR', 'Malgun Gothic', 'AppleGothic'] if name in available), 'DejaVu Sans')
plt.rcParams.update({'font.family': font_name, 'axes.unicode_minus': False, 'figure.dpi': 120})
pd.set_option('display.max_columns', 20)
pd.set_option('display.precision', 8)

WAFER_KEY = ['root_lot_id', 'wafer_id']
REQUIRED = ['part_id', 'root_lot_id', 'wafer_id', 'y_value', 'step_seq', 'eqp_id']
OPTIONAL = ['chamber_id', 'step_ord', 'tkin_time', 'line_id']

def part_heading(part_id):
    display(HTML('<h3>part_id = ' + html.escape(str(part_id)) + '</h3>'))

## 1. 실제 CSV 읽기
설비 이력에 같은 웨이퍼가 여러 번 등장해도 part별 전체 평균에는 한 번만 들어갑니다.
동일 part·웨이퍼에 서로 다른 y가 있으면 오류로 알립니다. y를 임의로 평균내거나 첫 값으로 덮지 않습니다.
part/웨이퍼 식별자가 없는 행과 raw y가 전혀 없는 웨이퍼는 건수를 표시하고 제외합니다.

In [ ]:
def read_raw_csv(path):
    path = Path(path).expanduser().resolve()
    if not path.is_file():
        raise FileNotFoundError(f'실제 raw.csv를 찾을 수 없습니다: {path}\nCSV_PATH를 실제 파일 경로로 지정하세요.')
    columns = pd.read_csv(path, nrows=0).columns
    missing = sorted(set(REQUIRED) - set(columns))
    if missing:
        raise ValueError(f'CSV에 필요한 컬럼이 없습니다: {missing}')
    if LINE_ID is not None and 'line_id' not in columns:
        raise ValueError('LINE_ID를 지정했지만 CSV에 line_id 컬럼이 없습니다.')
    use = REQUIRED + [c for c in OPTIONAL if c in columns]
    raw = pd.read_csv(path, usecols=use, dtype='string')
    rows_read = len(raw)
    for col in use:
        raw[col] = raw[col].str.strip().replace('', pd.NA)
    if LINE_ID is not None:
        raw = raw.loc[raw['line_id'] == str(LINE_ID)].copy()
    missing_id = raw[['part_id'] + WAFER_KEY].isna().any(axis=1)
    removed_id_rows = int(missing_id.sum())
    raw = raw.loc[~missing_id].copy()
    raw['_y'] = pd.to_numeric(raw['y_value'], errors='coerce').to_numpy(dtype=float, na_value=np.nan)
    raw.loc[~np.isfinite(raw['_y']), '_y'] = np.nan
    raw['_row'] = np.arange(len(raw))
    raw['chamber_id'] = raw['chamber_id'].fillna('') if 'chamber_id' in raw else ''
    raw['_time'] = pd.to_datetime(raw['tkin_time'], errors='coerce', utc=True) if 'tkin_time' in raw else pd.NaT
    fallback = pd.to_numeric(raw['step_ord'], errors='coerce') if 'step_ord' in raw else pd.Series(np.nan, index=raw.index)
    if STEP_SUFFIX:
        suffix = pd.to_numeric(raw['step_seq'].str.extract(STEP_SUFFIX, expand=False), errors='coerce')
        raw['_step'] = raw['step_seq'].str.replace(STEP_SUFFIX, '', regex=True)
        raw['_order'] = suffix.fillna(fallback)
    else:
        raw['_step'] = raw['step_seq']
        raw['_order'] = fallback
    raw['_order'] = raw['_order'].to_numpy(dtype=float, na_value=np.nan)
    found_parts = sorted(raw['part_id'].dropna().unique())
    if PART_IDS is None:
        selected = found_parts
    else:
        requested = [str(PART_IDS)] if isinstance(PART_IDS, (str, int)) else list(dict.fromkeys(map(str, PART_IDS)))
        unknown = sorted(set(requested) - set(found_parts))
        if unknown:
            raise ValueError(f'CSV에 없는 part_id: {unknown}')
        selected = requested
    if not selected:
        raise ValueError('분석할 part가 없습니다. CSV_PATH, LINE_ID, PART_IDS를 확인하세요.')
    print(f'실제 데이터: {path}')
    print(f'읽은 행 {rows_read:,} · 식별자 결측 제외 행 {removed_id_rows:,} · 분석 part {len(selected):,}개')
    return raw, selected

raw, selected_parts = read_raw_csv(CSV_PATH)
display(raw.groupby('part_id', observed=True).size().reindex(selected_parts).rename('이력 행 수').to_frame())

## 2. 실제 경로 조합의 raw 평균 계산
같은 STEP 안의 Order를 1개부터 `MAX_DEPTH`개까지 묶고, 실제로 관측된 설비·챔버 조합을 모두 계산합니다.
표본 수가 `MIN_N` 이상인 조합을 표시합니다. 전체 평균의 기준 집단은 해당 part의 raw y가 있는 모든 고유 웨이퍼입니다.
`STEP`은 표시할 공정 조합 범위를 정하며, 그 범위의 모든 점으로 n별 표준편차와 타점 수를 계산합니다.

In [ ]:
POINT_COLUMNS = ['part_id', 'step', 'depth', 'path', 'n', 'sample_mean']

def prepare_part(part_raw, part_id):
    wafer_values = part_raw.groupby(WAFER_KEY, sort=True, observed=True)['_y'].agg(['first', 'nunique'])
    conflicts = wafer_values.loc[wafer_values['nunique'] > 1]
    if len(conflicts):
        examples = list(conflicts.index[:5])
        raise ValueError(f'part {part_id}: 동일 웨이퍼에 서로 다른 raw y가 있습니다 ({len(conflicts)}장). 예: {examples}')
    missing_y = int((wafer_values['nunique'] == 0).sum())
    wafer_values = wafer_values.loc[wafer_values['nunique'] == 1, ['first']].rename(columns={'first': 'raw_y'})
    if wafer_values.empty:
        raise ValueError(f'part {part_id}: 유효한 raw y가 있는 웨이퍼가 없습니다.')
    wf_index = wafer_values.index
    part_rows = part_raw.copy()
    part_rows['_wi'] = wf_index.get_indexer(pd.MultiIndex.from_frame(part_rows[WAFER_KEY]))
    valid_history = ((part_rows['_wi'] >= 0) & part_rows['_step'].notna() &
                     np.isfinite(part_rows['_order']) & part_rows['eqp_id'].notna())
    history = part_rows.loc[valid_history].copy()
    if STEP is not None:
        target_step = re.sub(STEP_SUFFIX, '', str(STEP)) if STEP_SUFFIX else str(STEP)
        history = history.loc[history['_step'] == target_step].copy()
    history = history.sort_values(['_time', '_row'], kind='stable', na_position='first' if REWORK == 'last' else 'last')
    before = len(history)
    history = history.drop_duplicates(['_wi', '_step', '_order'], keep=REWORK)
    return wafer_values, history, {'missing_y_wafers': missing_y, 'duplicate_history_rows': before - len(history)}

def observed_combo_means(wafer_values, history, part_id):
    y = wafer_values['raw_y'].to_numpy(float)
    points = []
    for step, h in history.groupby('_step', sort=True, observed=True):
        orders = sorted(h['_order'].unique())
        order_index = {order: i for i, order in enumerate(orders)}
        # 설비와 챔버의 쌍 자체를 그룹 키로 사용한다.
        unit_pairs = pd.MultiIndex.from_frame(h[['eqp_id', 'chamber_id']])
        unit_codes, units = pd.factorize(unit_pairs, sort=True)
        assignment = np.full((len(y), len(orders)), -1, dtype=np.int32)
        assignment[h['_wi'].to_numpy(int), h['_order'].map(order_index).to_numpy(int)] = unit_codes
        for depth in range(1, min(MAX_DEPTH, len(orders)) + 1):
            for positions in combinations(range(len(orders)), depth):
                observed = assignment[:, list(positions)]
                present = (observed >= 0).all(axis=1)
                if int(present.sum()) < MIN_N:
                    continue
                routes, inverse, counts = np.unique(observed[present], axis=0, return_inverse=True, return_counts=True)
                sums = np.bincount(inverse, weights=y[present], minlength=len(routes))
                for route_id in np.flatnonzero(counts >= MIN_N):
                    route = routes[route_id]
                    labels = []
                    for position, unit_code in zip(positions, route):
                        order = orders[position]
                        order_label = str(int(order)) if float(order).is_integer() else str(order)
                        eqp, chamber = units[int(unit_code)]
                        unit_label = str(eqp) + (f' / {chamber}' if chamber else '')
                        labels.append(f'O{order_label}: {unit_label}')
                    n = int(counts[route_id])
                    points.append({'part_id': part_id, 'step': str(step), 'depth': depth,
                                   'path': ' → '.join(labels), 'n': n,
                                   'sample_mean': float(sums[route_id] / n)})
    return pd.DataFrame(points, columns=POINT_COLUMNS)

def empirical_envelope(points, mu_hat):
    # sigma_hat_n은 n이 같은 평균 타점 자체의 표준편차이다.
    if points.empty:
        summary = pd.DataFrame(columns=['n', 'm_n', 'mean_of_means', 'sigma_hat_n', 'mu_hat', 'lower', 'upper'])
        result = points.copy()
        for col in ['m_n', 'sigma_hat_n', 'mu_hat', 'lower', 'upper']:
            result[col] = pd.Series(dtype=float)
        result['outside'] = pd.Series(dtype='boolean')
        result['direction'] = pd.Series(dtype=str)
        return result, summary
    summary = (points.groupby('n', sort=True)['sample_mean']
               .agg(m_n='size', mean_of_means='mean', sigma_hat_n=lambda x: x.std(ddof=1))
               .reset_index())
    summary['mu_hat'] = mu_hat
    half_width = summary['sigma_hat_n'] * np.sqrt(2 * np.log(summary['m_n']))
    summary['lower'] = mu_hat - half_width
    summary['upper'] = mu_hat + half_width
    result = points.merge(summary.drop(columns='mean_of_means'), on='n', how='left', validate='many_to_one')
    available = result['sigma_hat_n'].notna()
    result['outside'] = pd.Series(pd.NA, index=result.index, dtype='boolean')
    # 경계와 소수점 연산 오차 수준으로 같은 값은 경계 위로 취급한다. 경계 수식 자체는 바꾸지 않는다.
    scale = result[['sample_mean', 'lower', 'upper']].abs().max(axis=1)
    rounding_tolerance = 32 * np.finfo(float).eps * scale
    above = result['sample_mean'] > result['upper'] + rounding_tolerance
    below = result['sample_mean'] < result['lower'] - rounding_tolerance
    result.loc[available, 'outside'] = above.loc[available] | below.loc[available]
    result['direction'] = np.select([~available, above, below],
                                    ['경계 없음', '위쪽', '아래쪽'], default='경계 안')
    return result, summary

def analyze_part(part_raw, part_id):
    wafer_values, history, quality = prepare_part(part_raw, part_id)
    mu_hat = float(wafer_values['raw_y'].mean())
    points = observed_combo_means(wafer_values, history, part_id)
    points, envelope = empirical_envelope(points, mu_hat)
    return {'part_id': part_id, 'wafer_values': wafer_values, 'mu_hat': mu_hat,
            'points': points, 'envelope': envelope, 'quality': quality}

In [ ]:
results_by_part = {}
rows = []
for part_id in selected_parts:
    report = analyze_part(raw.loc[raw['part_id'] == part_id], part_id)
    results_by_part[part_id] = report
    points = report['points']
    rows.append({'part_id': part_id, '고유 웨이퍼 수': len(report['wafer_values']),
                 'raw y 평균': report['mu_hat'],
                 'raw y 표준편차': float(report['wafer_values']['raw_y'].std(ddof=1)),
                 '평균 타점 수': len(points), '경계 밖': int(points['outside'].fillna(False).sum()),
                 '경계 없음': int(points['outside'].isna().sum()),
                 'y 결측 제외 웨이퍼': report['quality']['missing_y_wafers'],
                 '중복 이력 정리 행': report['quality']['duplicate_history_rows']})
    print(f"part {part_id}: raw y 평균 = {report['mu_hat']:.10g}, 웨이퍼 {len(report['wafer_values']):,}장, 타점 {len(points):,}개")
part_summary = pd.DataFrame(rows).set_index('part_id')
display(part_summary)

## 3. part별 차트
점과 중심선은 모두 raw y 단위입니다. 점의 위치를 판정 기준에 맞춰 옮기지 않습니다.
위 패널은 조합 평균과 envelope, 아래 패널은 실제 n별 평균 타점 수입니다.

In [ ]:
def plot_part(report):
    points, summary = report['points'], report['envelope']
    part_id, mu_hat = report['part_id'], report['mu_hat']
    if points.empty:
        print(f'part {part_id}: MIN_N / MAX_DEPTH / STEP 조건에 맞는 관측 조합이 없습니다.')
        return None
    fig, (ax, axm) = plt.subplots(2, 1, figsize=(13, 7), sharex=True,
                                 gridspec_kw={'height_ratios': [4, 1], 'hspace': 0.08})
    inside = points['outside'].eq(False).fillna(False).to_numpy(bool)
    outside = points['outside'].fillna(False).to_numpy(bool)
    unavailable = points['outside'].isna().to_numpy(bool)
    ax.scatter(points.loc[inside, 'n'], points.loc[inside, 'sample_mean'], s=10,
               alpha=0.25, color='#65758b', label=f'경계 안 {inside.sum():,}개')
    ax.scatter(points.loc[outside, 'n'], points.loc[outside, 'sample_mean'], s=24,
               alpha=0.9, color='#d44838', label=f'경계 밖 {outside.sum():,}개', zorder=4)
    if unavailable.any():
        ax.scatter(points.loc[unavailable, 'n'], points.loc[unavailable, 'sample_mean'], s=22,
                   marker='x', color='#979797', label=f'm_n = 1 · 경계 없음 {unavailable.sum():,}개')
    x = summary['n'].to_numpy(float)
    lower, upper = summary['lower'].to_numpy(float), summary['upper'].to_numpy(float)
    ax.fill_between(x, lower, upper, color='#2789d2', alpha=0.10)
    ax.plot(x, upper, '.-', color='#2789d2', linewidth=1.3, markersize=3,
            label=r'$\hat{\mu}_p \pm \hat{\sigma}_{p,n}\sqrt{2\ln m_{p,n}}$')
    ax.plot(x, lower, '.-', color='#2789d2', linewidth=1.3, markersize=3)
    ax.axhline(mu_hat, color='#202b3b', linestyle='--', linewidth=1.3,
               label=f'part raw y 평균 = {mu_hat:.8g}')
    ax.set_title(f'part_id = {part_id}  |  raw y 평균 · 실제 관측 조합 {len(points):,}개', loc='left', pad=14)
    ax.set_ylabel('조합의 raw y 평균')
    ax.ticklabel_format(axis='y', style='plain', useOffset=False)
    ax.legend(loc='upper left', bbox_to_anchor=(1.01, 1), frameon=False)
    axm.vlines(x, 0, summary['m_n'].to_numpy(), color='#344964', linewidth=1)
    axm.scatter(x, summary['m_n'], color='#344964', s=6)
    axm.set(xlabel='조합의 고유 웨이퍼 수 n', ylabel='타점 수 m_n')
    axm.set_ylim(bottom=0)
    if LOG_X:
        axm.set_xscale('log')
        axm.xaxis.set_major_formatter(FuncFormatter(lambda value, _: f'{value:g}'))
    for a in (ax, axm):
        a.grid(alpha=0.2)
        a.spines[['top', 'right']].set_visible(False)
    fig.subplots_adjust(right=0.72, left=0.10, top=0.90, bottom=0.09)
    plt.show()
    return fig

for part_id, report in results_by_part.items():
    part_heading(part_id)
    plot_part(report)

## 4. n별 계산 값과 경계 밖 조합
`mean_of_means`는 같은 n의 평균 타점들을 다시 평균낸 값입니다. 경계의 중심 `mu_hat`은 전체 고유 웨이퍼의 raw 평균입니다.
어느 공정·경로가 경계 밖에 있는지 원래 값 그대로 확인할 수 있습니다.

In [ ]:
for part_id, report in results_by_part.items():
    part_heading(part_id)
    display(report['envelope'])
    points = report['points']
    outside = points.loc[points['outside'].fillna(False)].copy()
    if outside.empty:
        print('경계 밖 조합이 없습니다.')
    else:
        outside['raw_mean_difference'] = outside['sample_mean'] - report['mu_hat']
        outside = outside.loc[outside['raw_mean_difference'].abs().sort_values(ascending=False).index]
        display(outside[['step', 'depth', 'path', 'n', 'sample_mean', 'mu_hat',
                         'm_n', 'sigma_hat_n', 'lower', 'upper', 'direction']].head(OUTSIDE_ROWS))

## 선택: 결과 저장 / 제품 하나 다시 보기
설정의 `SAVE_DIR`를 지정하면 제품별 평균 타점과 envelope 계산표를 CSV로 저장합니다.
아래 `PART_TO_VIEW`를 지정하면 이미 계산한 해당 제품의 차트만 다시 표시합니다.

In [ ]:
if SAVE_DIR is not None:
    save_dir = Path(SAVE_DIR).expanduser()
    save_dir.mkdir(parents=True, exist_ok=True)
    part_summary.to_csv(save_dir / 'part_summary.csv', encoding='utf-8-sig')
    for part_id, report in results_by_part.items():
        safe = re.sub(r'[^\w.-]+', '_', str(part_id))[:80] or 'part'
        suffix = hashlib.sha256(str(part_id).encode()).hexdigest()[:8]
        stem = f'{safe}_{suffix}'
        report['points'].to_csv(save_dir / f'{stem}_means.csv', index=False, encoding='utf-8-sig')
        report['envelope'].to_csv(save_dir / f'{stem}_envelope.csv', index=False, encoding='utf-8-sig')
    print(f'저장 위치: {save_dir.resolve()}')

PART_TO_VIEW = None  # 예: 'PRODUCT_A'
if PART_TO_VIEW is not None:
    plot_part(results_by_part[str(PART_TO_VIEW)])